# Decision Trees - Coding Practice

Twelve questions of the kind asked in ML interviews, each with a worked answer.

In [8]:
# !pip install matplotlib
# !pip install scikit-learn 

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 4.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 3.7 MB/s eta 0:00:0000:0100:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn] [scikit-learn]


In [10]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_diabetes, make_classification
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ModuleNotFoundError: No module named 'sklearn'

## Q1. Implement Gini impurity and entropy from scratch

Write two functions that take an array of class labels and return the node's impurity. Handle the
empty node and the single-class node without producing `nan`.

**What is being checked:** that you know impurity depends only on the label distribution - not on
the features, not on the sample size - and that you guard `log(0)`.

In [ ]:
class ImpurityMeasures:
    """Impurity functions that score how mixed a set of labels is."""

    LOG_EPSILON = 1e-12

    @staticmethod
    def class_probabilities(labels):
        _, counts = np.unique(labels, return_counts=True)
        return counts / counts.sum()

    @classmethod
    def gini(cls, labels):
        if len(labels) == 0:
            return 0.0
        probabilities = cls.class_probabilities(labels)
        return float(1.0 - np.sum(probabilities ** 2))

    @classmethod
    def entropy(cls, labels):
        if len(labels) == 0:
            return 0.0
        probabilities = cls.class_probabilities(labels)
        # The epsilon keeps log2(0) from producing nan when a class is absent.
        return float(-np.sum(probabilities * np.log2(probabilities + cls.LOG_EPSILON)))


example_nodes = {
    "pure": np.array([1, 1, 1, 1]),
    "balanced": np.array([0, 0, 1, 1]),
    "skewed": np.array([0, 0, 0, 1]),
    "three_class": np.array([0, 1, 2]),
}

for name, labels in example_nodes.items():
    print(f"{name:12s} gini={ImpurityMeasures.gini(labels):.4f}  "
          f"entropy={ImpurityMeasures.entropy(labels):.4f}")

## Q2. Compute the information gain of a split

Given the parent labels and the two child label arrays, return how much impurity the split removed.

**What is being checked:** the weighting. Averaging the two children's impurity without weighting
by their sample counts is the single most common mistake, and it makes a tiny pure child look like
a brilliant split. The last line of the answer shows exactly that failure.

In [ ]:
class SplitScorer:
    """Scores a candidate split by how much impurity it removes."""

    @staticmethod
    def weighted_child_impurity(left_labels, right_labels, impurity_function):
        total_count = len(left_labels) + len(right_labels)
        if total_count == 0:
            return 0.0
        left_share = len(left_labels) / total_count
        right_share = len(right_labels) / total_count
        return (left_share * impurity_function(left_labels)
                + right_share * impurity_function(right_labels))

    @classmethod
    def information_gain(cls, parent_labels, left_labels, right_labels, impurity_function):
        parent_impurity = impurity_function(parent_labels)
        child_impurity = cls.weighted_child_impurity(left_labels, right_labels, impurity_function)
        return parent_impurity - child_impurity


parent = np.array([0, 0, 0, 0, 1, 1, 1, 1])
clean_left, clean_right = np.array([0, 0, 0, 0]), np.array([1, 1, 1, 1])
messy_left, messy_right = np.array([0, 0, 0, 1]), np.array([0, 1, 1, 1])

print("perfect split gain:",
      round(SplitScorer.information_gain(parent, clean_left, clean_right, ImpurityMeasures.gini), 4))
print("messy split gain:  ",
      round(SplitScorer.information_gain(parent, messy_left, messy_right, ImpurityMeasures.gini), 4))

# The weighting is what stops a tiny pure child from looking like a great split.
tiny_left, big_right = np.array([0]), np.array([0, 0, 0, 1, 1, 1, 1])
print("tiny pure child:   ",
      round(SplitScorer.information_gain(parent, tiny_left, big_right, ImpurityMeasures.gini), 4))

## Q3. Find the best split in a dataset

Scan every feature and every candidate threshold and return `(feature_index, threshold, gain)`.
Respect a `min_samples_leaf` constraint.

**What is being checked:** that you only test midpoints between consecutive *distinct* values
(testing every raw value is wasteful and gives duplicate partitions), and that you skip splits
leaving a child too small. The answer verifies the root split against scikit-learn.

In [ ]:
class BestSplitFinder:
    """Exhaustive scan over every feature and every candidate threshold."""

    @staticmethod
    def candidate_thresholds(feature_column):
        unique_values = np.unique(feature_column)
        if len(unique_values) < 2:
            return np.empty(0)
        # Only midpoints between consecutive distinct values can change the partition.
        return (unique_values[:-1] + unique_values[1:]) / 2.0

    @classmethod
    def find_best_split(cls, features, labels, impurity_function, min_samples_leaf=1):
        best_gain = -np.inf
        best_feature_index = None
        best_threshold = None

        for feature_index in range(features.shape[1]):
            feature_column = features[:, feature_index]
            for threshold in cls.candidate_thresholds(feature_column):
                left_mask = feature_column <= threshold
                right_mask = ~left_mask
                if left_mask.sum() < min_samples_leaf or right_mask.sum() < min_samples_leaf:
                    continue
                gain = SplitScorer.information_gain(
                    labels, labels[left_mask], labels[right_mask], impurity_function
                )
                if gain > best_gain:
                    best_gain = gain
                    best_feature_index = feature_index
                    best_threshold = threshold

        return best_feature_index, best_threshold, best_gain


iris = load_iris()
feature_index, threshold, gain = BestSplitFinder.find_best_split(
    iris.data, iris.target, ImpurityMeasures.gini
)
print(f"best root split: {iris.feature_names[feature_index]} <= {threshold:.3f}  (gain {gain:.4f})")

reference_tree = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_SEED).fit(iris.data, iris.target)
print(f"sklearn root  : {iris.feature_names[reference_tree.tree_.feature[0]]} "
      f"<= {reference_tree.tree_.threshold[0]:.3f}")

## Q4. Build a decision tree classifier from scratch

Put Q1-Q3 together into a class with `fit` and `predict`, supporting `max_depth`,
`min_samples_split`, `min_samples_leaf` and `criterion`.

**What is being checked:** the recursion and its base cases. Name them out loud as you write:
depth limit reached, too few samples to split, node already pure, and no split produces positive
gain. Forgetting the last one grows useless branches forever.

In [ ]:
class TreeNode:
    """Either a split node (feature, threshold, two children) or a leaf (a prediction)."""

    def __init__(self, prediction=None, feature_index=None, threshold=None, left=None, right=None):
        self.prediction = prediction
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right

    @property
    def is_leaf(self):
        return self.left is None and self.right is None


class DecisionTreeClassifierScratch:
    """CART-style classifier grown by greedy recursive binary splitting."""

    def __init__(self, max_depth=5, min_samples_split=2, min_samples_leaf=1, criterion="gini"):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.root = None

    @property
    def impurity_function(self):
        return ImpurityMeasures.entropy if self.criterion == "entropy" else ImpurityMeasures.gini

    @staticmethod
    def majority_class(labels):
        values, counts = np.unique(labels, return_counts=True)
        return values[np.argmax(counts)]

    def fit(self, features, labels):
        features = np.asarray(features, dtype=float)
        labels = np.asarray(labels)
        self.root = self._grow(features, labels, depth=0)
        return self

    def _should_stop(self, labels, depth):
        if depth >= self.max_depth:
            return True
        if len(labels) < self.min_samples_split:
            return True
        return len(np.unique(labels)) == 1

    def _grow(self, features, labels, depth):
        if self._should_stop(labels, depth):
            return TreeNode(prediction=self.majority_class(labels))

        feature_index, threshold, gain = BestSplitFinder.find_best_split(
            features, labels, self.impurity_function, self.min_samples_leaf
        )
        # No split helps, so stop here rather than growing a useless branch.
        if feature_index is None or gain <= 0:
            return TreeNode(prediction=self.majority_class(labels))

        left_mask = features[:, feature_index] <= threshold
        right_mask = ~left_mask
        return TreeNode(
            feature_index=feature_index,
            threshold=threshold,
            left=self._grow(features[left_mask], labels[left_mask], depth + 1),
            right=self._grow(features[right_mask], labels[right_mask], depth + 1),
        )

    def predict(self, features):
        features = np.asarray(features, dtype=float)
        return np.array([self._predict_one(row) for row in features])

    def _predict_one(self, row):
        node = self.root
        while not node.is_leaf:
            node = node.left if row[node.feature_index] <= node.threshold else node.right
        return node.prediction

    def depth(self, node=None):
        node = self.root if node is None else node
        if node.is_leaf:
            return 0
        return 1 + max(self.depth(node.left), self.depth(node.right))


features_train, features_test, labels_train, labels_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=RANDOM_SEED, stratify=iris.target
)

scratch_tree = DecisionTreeClassifierScratch(max_depth=4).fit(features_train, labels_train)
print("scratch test accuracy:", round(accuracy_score(labels_test, scratch_tree.predict(features_test)), 4))
print("grown depth:", scratch_tree.depth())

## Q5. Check it against scikit-learn

Train both implementations at several depths on the same split and compare accuracy.

**What is being checked:** whether you validate your own work. Expect close but not identical
numbers - ties between equally good splits are broken differently, and scikit-learn permutes
feature order using `random_state`. If your numbers are *wildly* off, the usual culprit is
unweighted child impurity from Q2.

In [ ]:
comparison_rows = []
for depth in [1, 2, 3, 4, 5, None]:
    depth_limit = depth if depth is not None else 20
    scratch_model = DecisionTreeClassifierScratch(max_depth=depth_limit).fit(features_train, labels_train)
    sklearn_model = DecisionTreeClassifier(
        max_depth=depth, criterion="gini", random_state=RANDOM_SEED
    ).fit(features_train, labels_train)

    comparison_rows.append((
        str(depth),
        accuracy_score(labels_test, scratch_model.predict(features_test)),
        accuracy_score(labels_test, sklearn_model.predict(features_test)),
    ))

print(f"{'max_depth':>10s}  {'scratch':>8s}  {'sklearn':>8s}")
for depth_label, scratch_accuracy, sklearn_accuracy in comparison_rows:
    print(f"{depth_label:>10s}  {scratch_accuracy:>8.4f}  {sklearn_accuracy:>8.4f}")

## Q6. Turn it into a regression tree

Adapt the classifier to predict a continuous target.

**What is being checked:** that you know only **two** things change - the split criterion becomes
variance reduction instead of Gini, and the leaf returns the mean instead of the majority class.
The growth procedure, the threshold search and the stopping rules are all identical. Say that in
the interview; it shows you understand the algorithm rather than having memorised two of them.

In [ ]:
class DecisionTreeRegressorScratch:
    """Same growth procedure as the classifier: only the criterion and the leaf rule change."""

    def __init__(self, max_depth=5, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root = None

    @staticmethod
    def variance(values):
        if len(values) == 0:
            return 0.0
        return float(np.var(values))

    @staticmethod
    def leaf_value(values):
        return float(np.mean(values))

    def fit(self, features, targets):
        features = np.asarray(features, dtype=float)
        targets = np.asarray(targets, dtype=float)
        self.root = self._grow(features, targets, depth=0)
        return self

    def _should_stop(self, targets, depth):
        if depth >= self.max_depth:
            return True
        if len(targets) < self.min_samples_split:
            return True
        return bool(np.allclose(targets, targets[0]))

    def _grow(self, features, targets, depth):
        if self._should_stop(targets, depth):
            return TreeNode(prediction=self.leaf_value(targets))

        feature_index, threshold, gain = BestSplitFinder.find_best_split(
            features, targets, self.variance, self.min_samples_leaf
        )
        if feature_index is None or gain <= 0:
            return TreeNode(prediction=self.leaf_value(targets))

        left_mask = features[:, feature_index] <= threshold
        right_mask = ~left_mask
        return TreeNode(
            feature_index=feature_index,
            threshold=threshold,
            left=self._grow(features[left_mask], targets[left_mask], depth + 1),
            right=self._grow(features[right_mask], targets[right_mask], depth + 1),
        )

    def predict(self, features):
        features = np.asarray(features, dtype=float)
        return np.array([self._predict_one(row) for row in features])

    def _predict_one(self, row):
        node = self.root
        while not node.is_leaf:
            node = node.left if row[node.feature_index] <= node.threshold else node.right
        return node.prediction


diabetes = load_diabetes()
regression_train, regression_test, target_train, target_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.3, random_state=RANDOM_SEED
)

scratch_regressor = DecisionTreeRegressorScratch(max_depth=3).fit(regression_train, target_train)
sklearn_regressor = DecisionTreeRegressor(max_depth=3, random_state=RANDOM_SEED).fit(
    regression_train, target_train
)

print("scratch MSE:", round(mean_squared_error(target_test, scratch_regressor.predict(regression_test)), 2))
print("sklearn MSE:", round(mean_squared_error(target_test, sklearn_regressor.predict(regression_test)), 2))

## Q7. Show the overfitting curve against max_depth

Train trees at `max_depth` 1 through 20 and plot train vs holdout accuracy.

**What is being checked:** that you can *demonstrate* the low-bias/high-variance claim rather than
just assert it. Note where train accuracy hits 1.0 while holdout accuracy is already falling - that
gap is the whole argument for pruning.

In [ ]:
signal_features, signal_labels = make_classification(
    n_samples=1200, n_features=20, n_informative=6, n_redundant=2,
    class_sep=0.8, flip_y=0.05, random_state=RANDOM_SEED
)
depth_train, depth_holdout, depth_train_labels, depth_holdout_labels = train_test_split(
    signal_features, signal_labels, test_size=0.3, random_state=RANDOM_SEED
)

depth_grid = range(1, 21)
train_scores = []
holdout_scores = []
for depth in depth_grid:
    model = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_SEED)
    model.fit(depth_train, depth_train_labels)
    train_scores.append(model.score(depth_train, depth_train_labels))
    holdout_scores.append(model.score(depth_holdout, depth_holdout_labels))

best_depth = list(depth_grid)[int(np.argmax(holdout_scores))]
print(f"best max_depth on holdout: {best_depth}  (accuracy {max(holdout_scores):.4f})")
print(f"train accuracy at that depth: {train_scores[best_depth - 1]:.4f}")
print(f"train accuracy fully grown  : {train_scores[-1]:.4f}  "
      f"holdout {holdout_scores[-1]:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(list(depth_grid), train_scores, marker="o", label="train")
plt.plot(list(depth_grid), holdout_scores, marker="o", label="holdout")
plt.axvline(best_depth, linestyle="--", color="grey", label=f"best depth = {best_depth}")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.title("A tree memorises the training set as depth grows")
plt.legend()
plt.tight_layout()
plt.show()

## Q8. Prune with cost-complexity pruning

Use `cost_complexity_pruning_path` to get the alpha candidates, cross-validate over them, and
compare the pruned tree against the unpruned one on node count and holdout accuracy.

**What is being checked:** whether you have actually used post-pruning, not just heard of it. The
result to point at: far fewer nodes *and* better holdout accuracy.

In [ ]:
full_tree = DecisionTreeClassifier(random_state=RANDOM_SEED)
pruning_path = full_tree.cost_complexity_pruning_path(depth_train, depth_train_labels)
alphas = pruning_path.ccp_alphas[:-1]          # drop the alpha that collapses the tree to one leaf
alphas = alphas[alphas > 0]

alpha_scores = []
for alpha in alphas[::5]:                       # subsample the path to keep this quick
    model = DecisionTreeClassifier(ccp_alpha=alpha, random_state=RANDOM_SEED)
    mean_score = cross_val_score(model, depth_train, depth_train_labels, cv=5).mean()
    alpha_scores.append((alpha, mean_score))

best_alpha, best_score = max(alpha_scores, key=lambda pair: pair[1])
print(f"best ccp_alpha: {best_alpha:.5f}  (cv accuracy {best_score:.4f})")

unpruned = DecisionTreeClassifier(random_state=RANDOM_SEED).fit(depth_train, depth_train_labels)
pruned = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=RANDOM_SEED).fit(
    depth_train, depth_train_labels
)
print(f"unpruned: {unpruned.tree_.node_count:4d} nodes, holdout "
      f"{unpruned.score(depth_holdout, depth_holdout_labels):.4f}")
print(f"pruned  : {pruned.tree_.node_count:4d} nodes, holdout "
      f"{pruned.score(depth_holdout, depth_holdout_labels):.4f}")

## Q9. Show that impurity-based feature importance is biased

Build a dataset where the real drivers are binary and add one pure-noise continuous column. Fit a
tree and compare `feature_importances_` (MDI) against permutation importance.

**What is being checked:** the most useful thing you can know about tree feature importance. MDI is
computed on training data and rewards features with many candidate split points, so a high-cardinality
noise column collects a large share of the credit. Permutation importance is measured on held-out
data and scores it near zero. This is the answer to "how would you pick features from a tree?".

In [ ]:
# The bias only shows when the honest features have FEW split points and the junk
# feature has MANY. Three binary drivers plus one continuous pure-noise column.
bias_generator = np.random.RandomState(RANDOM_SEED)
sample_count = 2000

binary_features = bias_generator.randint(0, 2, size=(sample_count, 3))
clean_labels = (binary_features.sum(axis=1) >= 2).astype(int)
label_flips = bias_generator.rand(sample_count) < 0.10
bias_labels = np.where(label_flips, 1 - clean_labels, clean_labels)

# Pure noise, but a distinct value per row means thousands of candidate thresholds.
noise_feature = bias_generator.rand(sample_count, 1)
bias_features = np.hstack([binary_features, noise_feature])
bias_names = ["binary_a", "binary_b", "binary_c", "random_noise"]

bias_train, bias_holdout, bias_train_labels, bias_holdout_labels = train_test_split(
    bias_features, bias_labels, test_size=0.3, random_state=RANDOM_SEED
)
biased_tree = DecisionTreeClassifier(random_state=RANDOM_SEED).fit(bias_train, bias_train_labels)

impurity_importance = biased_tree.feature_importances_
permutation_result = permutation_importance(
    biased_tree, bias_holdout, bias_holdout_labels, n_repeats=10, random_state=RANDOM_SEED
)

print(f"{'feature':>14s}  {'MDI':>8s}  {'permutation':>12s}")
for index in np.argsort(impurity_importance)[::-1]:
    print(f"{bias_names[index]:>14s}  {impurity_importance[index]:>8.4f}  "
          f"{permutation_result.importances_mean[index]:>12.4f}")

print(f"\nrandom_noise carries no signal at all, yet MDI gives it "
      f"{impurity_importance[-1]:.1%} of the credit.")
print("Permutation importance, measured on held-out data, correctly scores it near zero.")

## Q10. Extract the decision rules from a fitted tree

Walk `tree_.children_left` / `children_right` / `feature` / `threshold` and print the
`IF ... AND ... THEN ...` rule for every leaf.

**What is being checked:** that "interpretable" means something concrete to you. One gotcha worth
knowing: modern scikit-learn stores class *proportions* in `tree_.value`, not raw counts, so
multiply by `n_node_samples` to recover counts.

In [ ]:
class RuleExtractor:
    """Walks a fitted scikit-learn tree and prints the path to every leaf."""

    LEAF_MARKER = -1  # sklearn stores -1 as the child index of a leaf

    @classmethod
    def print_rules(cls, fitted_tree, feature_names, class_names=None):
        structure = fitted_tree.tree_
        cls._walk(structure, node_index=0, conditions=[],
                  feature_names=feature_names, class_names=class_names)

    @classmethod
    def _walk(cls, structure, node_index, conditions, feature_names, class_names):
        left_child = structure.children_left[node_index]
        right_child = structure.children_right[node_index]

        if left_child == cls.LEAF_MARKER:
            samples = int(structure.n_node_samples[node_index])
            # Modern sklearn stores class PROPORTIONS in tree_.value, not raw counts.
            class_counts = np.round(structure.value[node_index][0] * samples).astype(int)
            predicted = int(np.argmax(class_counts))
            label = class_names[predicted] if class_names is not None else predicted
            print(f"IF {' AND '.join(conditions) if conditions else 'always'}")
            print(f"   THEN {label}  (samples={samples}, counts={class_counts.tolist()})")
            return

        feature_name = feature_names[structure.feature[node_index]]
        threshold = structure.threshold[node_index]
        cls._walk(structure, left_child, conditions + [f"{feature_name} <= {threshold:.2f}"],
                  feature_names, class_names)
        cls._walk(structure, right_child, conditions + [f"{feature_name} > {threshold:.2f}"],
                  feature_names, class_names)


rule_tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED).fit(iris.data, iris.target)
RuleExtractor.print_rules(rule_tree, iris.feature_names, iris.target_names)

## Q11. Show that a single tree is unstable

Take a dataset with two nearly-equally-informative features, refit on 40 bootstrap resamples, and
record which feature wins the root split each time.

**What is being checked:** the variance claim, made concrete. The root split flips between the two
twins depending on resampling noise, which is why "the top split is the most important feature" is
a claim you should refuse to make - and why averaging many trees helps.

In [ ]:
class InstabilityProbe:
    """Refits a tree on bootstrap resamples and records which feature wins the root split."""

    @staticmethod
    def root_feature(features, labels, feature_names):
        model = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_SEED)
        model.fit(features, labels)
        return feature_names[model.tree_.feature[0]]

    @classmethod
    def survey(cls, features, labels, feature_names, trials=40):
        generator = np.random.RandomState(RANDOM_SEED)
        chosen_roots = []
        for _ in range(trials):
            sample_index = generator.choice(len(labels), size=len(labels), replace=True)
            chosen_roots.append(
                cls.root_feature(features[sample_index], labels[sample_index], feature_names)
            )
        return chosen_roots


# Two features that carry almost the same information: the tree has to pick one,
# and which one it picks is decided by noise.
unstable_generator = np.random.RandomState(RANDOM_SEED)
hidden_signal = unstable_generator.randn(300)
unstable_features = np.column_stack([
    hidden_signal + 0.30 * unstable_generator.randn(300),
    hidden_signal + 0.30 * unstable_generator.randn(300),
    unstable_generator.randn(300),
])
unstable_labels = (hidden_signal > 0).astype(int)
unstable_names = ["twin_a", "twin_b", "distractor"]

roots = InstabilityProbe.survey(unstable_features, unstable_labels, unstable_names)
names, counts = np.unique(roots, return_counts=True)

print("root split feature across 40 bootstrap resamples of the SAME dataset:")
for index in np.argsort(counts)[::-1]:
    print(f"  {names[index]:>12s}  chosen {counts[index]:2d} / 40 times")
print(f"\n{len(names)} different features win the root split. The tree structure - and so any")
print("story you tell from 'the top split is the most important feature' - is not stable.")

## Q12. Show that a regression tree cannot extrapolate

Fit a regression tree on x in [0, 10] and ask it to predict out to x = 20.

**What is being checked:** that you know a tree's prediction is a step function over the training
range and a flat line beyond it. Every input past the last split falls into the same leaf and gets
the same constant. This is why trees are the wrong tool for a trending time series, and where a
linear model beats them outright.

In [ ]:
train_range = np.linspace(0, 10, 120).reshape(-1, 1)
train_signal = np.sin(train_range).ravel() + 0.4 * train_range.ravel()

extrapolation_tree = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED)
extrapolation_tree.fit(train_range, train_signal)

# Ask for predictions well outside the range the tree ever saw.
full_range = np.linspace(0, 20, 400).reshape(-1, 1)
predictions = extrapolation_tree.predict(full_range)

print("last training input :", round(float(train_range.max()), 2))
print("prediction at x=10  :", round(float(extrapolation_tree.predict([[10.0]])[0]), 3))
print("prediction at x=15  :", round(float(extrapolation_tree.predict([[15.0]])[0]), 3))
print("prediction at x=20  :", round(float(extrapolation_tree.predict([[20.0]])[0]), 3))
print("-> identical: beyond the last split the tree returns one constant leaf value")

plt.figure(figsize=(7, 4))
plt.scatter(train_range, train_signal, s=12, label="training data")
plt.plot(full_range, predictions, color="crimson", label="tree prediction")
plt.axvline(10, linestyle="--", color="grey", label="edge of training range")
plt.xlabel("x")
plt.ylabel("y")
plt.title("A regression tree is flat outside the range it was trained on")
plt.legend()
plt.tight_layout()
plt.show()